<a href="https://colab.research.google.com/github/safia-ahmad/calib-cl/blob/main/notebooks/05_validate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
%cd /content
!rm -rf calib-cl
!git clone https://github.com/safia-ahmad/calib-cl.git
%cd calib-cl
!ls calcl tests
!pip install -q pytest
!python -m pytest -q

/content
Cloning into 'calib-cl'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 115 (delta 48), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 51.86 KiB | 870.00 KiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/calib-cl
calcl:
alarms.py  __init__.py	run_stream_ILLUSTRATIVE.py  train.py
corp.py    metrics.py	splits.py		    validate.py
data.py    models.py	stream.py

tests:
test_alarms.py	test_data.py	 test_splits.py  test_train.py
test_corp.py	test_metrics.py  test_stream.py  test_validate.py
...............................                                          [100%]
31 passed in 14.00s


In [ ]:
%cd /content
!rm -rf calib-cl
!git clone -q https://github.com/safia-ahmad/calib-cl.git
%cd /content/calib-cl
import os, urllib.request
import pandas as pd
os.makedirs('/content/data/pcam_meta', exist_ok=True)
print("ready")

/content
/content/calib-cl
ready


In [ ]:
base = "https://zenodo.org/records/1494286/files/"
for s in ["train", "valid", "test"]:
    f = f"camelyonpatch_level_2_split_{s}_meta.csv"
    path = f"/content/data/pcam_meta/{f}"
    if not os.path.exists(path):
        urllib.request.urlretrieve(base + f + "?download=1", path)
    print(s, round(os.path.getsize(path) / 1e6, 1), "MB")

train 15.0 MB
valid 1.9 MB
test 1.6 MB


In [ ]:
meta = {s: pd.read_csv(f"/content/data/pcam_meta/camelyonpatch_level_2_split_{s}_meta.csv", index_col=0)
        for s in ["train", "valid", "test"]}
for s, df in meta.items():
    print(s, df.shape)
    display(df.head(3))

train (262144, 5)


,coord_y,coord_x,tumor_patch,center_tumor_patch,wsi
0,63104,43648,False,False,camelyon16_train_tumor_104
1,148544,74048,True,True,camelyon16_train_tumor_003
2,64192,78912,True,True,camelyon16_train_tumor_089


valid (32768, 5)


,coord_y,coord_x,tumor_patch,center_tumor_patch,wsi
0,26496,35136,True,True,camelyon16_train_tumor_085
1,96576,59520,True,True,camelyon16_train_tumor_069
2,19840,46848,True,True,camelyon16_train_tumor_097


test (32768, 5)


,coord_y,coord_x,tumor_patch,center_tumor_patch,wsi
0,145856,73984,False,False,camelyon16_test_032
1,92224,76032,True,True,camelyon16_test_029
2,14720,15936,False,False,camelyon16_test_055


In [ ]:
rows = []
for s, df in meta.items():
    per_slide = df.groupby("wsi").size()
    rows.append({
        "split": s,
        "patches": len(df),
        "slides": df.wsi.nunique(),
        "tumour_patch_fraction": round(df.center_tumor_patch.mean(), 3),
        "min_patches_per_slide": per_slide.min(),
        "median_patches_per_slide": int(per_slide.median()),
        "max_patches_per_slide": per_slide.max(),
    })
summary = pd.DataFrame(rows)
display(summary)

,split,patches,slides,tumour_patch_fraction,min_patches_per_slide,median_patches_per_slide,max_patches_per_slide
0,train,262144,216,0.5,537,697,3800
1,valid,32768,54,0.5,259,345,1728
2,test,32768,129,0.5,98,140,716


In [ ]:
sets = {s: set(df.wsi) for s, df in meta.items()}
for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    print(f"{a} & {b}: {len(sets[a] & sets[b])} shared slides")
print("total distinct slides:", len(set.union(*sets.values())))

train & valid: 0 shared slides
train & test: 0 shared slides
valid & test: 0 shared slides
total distinct slides: 399


In [ ]:
url = "https://camelyon-dataset.s3.amazonaws.com/CAMELYON16/evaluation/reference.csv"
urllib.request.urlretrieve(url, "/content/data/reference.csv")
ref = pd.read_csv("/content/data/reference.csv")
ref["split"] = ref.image.str.startswith("test").map({True: "test", False: "train"})
print(len(ref), "slides")
display(ref.head())
display(pd.crosstab([ref.split, ref.type], ref.center, margins=True))

399 slides


,image,type,class,center,split
0,normal_001.tif,normal,negative,0,train
1,normal_002.tif,normal,negative,0,train
2,normal_003.tif,normal,negative,0,train
3,normal_004.tif,normal,negative,0,train
4,normal_005.tif,normal,negative,0,train


center          0    1  All
split type                 
test  normal   50   30   80
      tumor    24   25   49
train normal  100   59  159
      tumor    71   40  111
All           245  154  399

In [ ]:
tab = pd.crosstab(ref.split, ref.center)
big = ref.center.value_counts().idxmax()          # the centre with more slides should be RUMC
small = [c for c in tab.columns if c != big][0]
got = {sp: (tab.loc[sp, big], tab.loc[sp, small]) for sp in ["train", "test"]}
expected = {"train": (170, 100), "test": (79, 50)}
print(f"code {big} = RUMC candidate, code {small} = UMCU candidate")
for sp in ["train", "test"]:
    print(sp, "got", got[sp], "expected", expected[sp], "OK" if got[sp] == expected[sp] else "MISMATCH")
print("G1:", "PASS" if got == expected else "CHECK, paste this output to Claude")

code 0 = RUMC candidate, code 1 = UMCU candidate
train got (np.int64(171), np.int64(99)) expected (170, 100) MISMATCH
test got (np.int64(74), np.int64(55)) expected (79, 50) MISMATCH
G1: CHECK, paste this output to Claude


In [ ]:
def wsi_to_image(w):
    p = w.split("_")                 # e.g. camelyon16_train_tumor_104
    return f"{p[-2]}_{p[-1]}.tif"    # -> tumor_104.tif

allm = pd.concat([df.assign(pcam_split=s) for s, df in meta.items()])
allm["image"] = allm.wsi.map(wsi_to_image)
m = allm.merge(ref[["image", "center", "type"]], on="image", how="left")
print("patches without a hospital:", m.center.isna().sum())
display(pd.crosstab(m.pcam_split, m.center, margins=True))
display(m.groupby("center").agg(slides=("wsi", "nunique"), patches=("wsi", "size"),
                                tumour_fraction=("center_tumor_patch", "mean")))

patches without a hospital: 1180


center,0.0,1.0,All
pcam_split,,,
test,16150,16175,32325
train,160752,100655,261407
valid,20521,12247,32768
All,197423,129077,326500


,slides,patches,tumour_fraction
center,,,
0.0,244,197423,0.472498
1.0,153,129077,0.544380


In [ ]:
# 1. Which slides had no hospital?
lost = m[m.center.isna()].groupby(["pcam_split", "wsi"]).size()
print("Slides missing from reference.csv:\n", lost, "\n")

# 2. The slides we will use: CAMELYON16 training slides only
use = ref[ref.split == "train"].copy()
use["hospital"] = use.center.map({0: "RUMC", 1: "UMCU"})
display(pd.crosstab(use.hospital, use.type, margins=True))

# 3. Patch supply per hospital and class (Gate G3)
pm = m[m.image.isin(use.image)].copy()
pm["hospital"] = pm.center.map({0: "RUMC", 1: "UMCU"})
display(pm.groupby(["hospital", "center_tumor_patch"]).size().unstack())
print("Slides that contain tumour patches:\n", pm[pm.center_tumor_patch].groupby("hospital").wsi.nunique())

# 4. Save the slide list
use.to_csv("/content/pcam_slides_used.csv", index=False)
from google.colab import files
files.download("/content/pcam_slides_used.csv")

Slides missing from reference.csv:
 pcam_split  wsi                        
test        camelyon16_test_049            443
train       camelyon16_train_normal_086    737
dtype: int64 



type,normal,tumor,All
hospital,,,
RUMC,100,71,171
UMCU,59,40,99
All,159,111,270


center_tumor_patch,False,True
hospital,,
RUMC,94540,86733
UMCU,52194,60708


Slides that contain tumour patches:
 hospital
RUMC    70
UMCU    40
Name: wsi, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%cd /content
!rm -rf calib-cl
!git clone -q https://github.com/safia-ahmad/calib-cl.git
from google.colab import drive
drive.mount('/content/drive')
import os, urllib.request
import numpy as np, pandas as pd
RAW = "/content/data/pcam_raw"
OUT = "/content/drive/MyDrive/calib-cl-data"
os.makedirs(RAW, exist_ok=True); os.makedirs(OUT, exist_ok=True)
!df -h /content | tail -1

/content


MessageError: Error: credential propagation was unsuccessful

In [ ]:
base = "https://zenodo.org/records/1494286/files/"
files_needed = ["camelyonpatch_level_2_split_train_x.h5.gz", "camelyonpatch_level_2_split_train_y.h5.gz",
                "camelyonpatch_level_2_split_valid_x.h5.gz", "camelyonpatch_level_2_split_valid_y.h5.gz"]
for f in files_needed:
    print("downloading", f)
    !wget -q -c --show-progress -O {RAW}/{f} "{base}{f}?download=1"
!ls -lh {RAW}

In [ ]:
!gunzip -f {RAW}/*.gz
!ls -lh {RAW}
import h5py
for s in ["train", "valid"]:
    with h5py.File(f"{RAW}/camelyonpatch_level_2_split_{s}_x.h5", "r") as fx, \
         h5py.File(f"{RAW}/camelyonpatch_level_2_split_{s}_y.h5", "r") as fy:
        print(s, "keys:", list(fx.keys()), list(fy.keys()), "x shape:", fx["x"].shape, fx["x"].dtype)
        y = fy["y"][:].reshape(-1)
    meta_s = pd.read_csv(f"/content/data/pcam_meta/camelyonpatch_level_2_split_{s}_meta.csv", index_col=0) \
        if os.path.exists(f"/content/data/pcam_meta/camelyonpatch_level_2_split_{s}_meta.csv") else None
    if meta_s is None:
        os.makedirs("/content/data/pcam_meta", exist_ok=True)
        fn = f"camelyonpatch_level_2_split_{s}_meta.csv"
        urllib.request.urlretrieve(base + fn + "?download=1", f"/content/data/pcam_meta/{fn}")
        meta_s = pd.read_csv(f"/content/data/pcam_meta/{fn}", index_col=0)
    agree = (y == meta_s.center_tumor_patch.astype(int).to_numpy()).mean()
    print(f"  labels agree with metadata: {agree:.4f}")

In [ ]:
CAP = 500
ref_path = "/content/data/reference.csv"
if not os.path.exists(ref_path):
    urllib.request.urlretrieve("https://camelyon-dataset.s3.amazonaws.com/CAMELYON16/evaluation/reference.csv", ref_path)
ref = pd.read_csv(ref_path)
use = ref[~ref.image.str.startswith("test")].copy()
use["hospital"] = use.center.map({0: "RUMC", 1: "UMCU"})
hosp = dict(zip(use.image, use.hospital))

def wsi_to_image(w):
    p = w.split("_")
    return f"{p[-2]}_{p[-1]}.tif"

parts = []
for s in ["train", "valid"]:
    df = pd.read_csv(f"/content/data/pcam_meta/camelyonpatch_level_2_split_{s}_meta.csv", index_col=0)
    df["pcam_split"], df["row"] = s, np.arange(len(df))
    df["image"] = df.wsi.map(wsi_to_image)
    df["hospital"] = df.image.map(hosp)
    parts.append(df[df.hospital.notna()])
meta = pd.concat(parts, ignore_index=True)

rng = np.random.default_rng(0)
keep = []
for wsi, g in meta.groupby("wsi"):
    idx = g.index.to_numpy()
    keep.append(idx if len(idx) <= CAP else rng.choice(idx, CAP, replace=False))
sel = meta.loc[np.concatenate(keep)].sort_values(["pcam_split", "row"]).reset_index(drop=True)
sel["label"] = sel.center_tumor_patch.astype(int)
print("patches kept:", len(sel), "| slides:", sel.wsi.nunique())

tmp = "/content/data/pcam_compact.h5"
N = len(sel)
with h5py.File(tmp, "w") as fo:
    X = fo.create_dataset("x", (N, 96, 96, 3), dtype="uint8", chunks=(64, 96, 96, 3), compression="gzip", compression_opts=4)
    Y = fo.create_dataset("y", (N,), dtype="uint8")
    pos = 0
    for s in ["train", "valid"]:
        rows = sel.loc[sel.pcam_split == s, "row"].to_numpy()
        with h5py.File(f"{RAW}/camelyonpatch_level_2_split_{s}_x.h5", "r") as fx, \
             h5py.File(f"{RAW}/camelyonpatch_level_2_split_{s}_y.h5", "r") as fy:
            B = 8192
            for start in range(0, fx["x"].shape[0], B):
                inb = rows[(rows >= start) & (rows < start + B)]
                if len(inb) == 0:
                    continue
                xb = fx["x"][start:start + B]
                yb = fy["y"][start:start + B].reshape(-1)
                X[pos:pos + len(inb)] = xb[inb - start]
                Y[pos:pos + len(inb)] = yb[inb - start]
                pos += len(inb)
    assert pos == N, (pos, N)
    assert (Y[:] == sel.label.to_numpy()).all(), "labels do not match"

!cp {tmp} {OUT}/pcam_compact.h5
sel.to_csv(f"{OUT}/pcam_compact_manifest.csv", index=False)
!ls -lh {OUT}

In [ ]:
import matplotlib.pyplot as plt
man = pd.read_csv(f"{OUT}/pcam_compact_manifest.csv")
display(man.groupby(["hospital", "label"]).agg(patches=("wsi", "size"), slides=("wsi", "nunique")))

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
with h5py.File(f"{OUT}/pcam_compact.h5", "r") as f:
    for r, (h, lab) in enumerate([("RUMC", 0), ("RUMC", 1), ("UMCU", 0), ("UMCU", 1)]):
        idx = np.sort(man[(man.hospital == h) & (man.label == lab)].sample(8, random_state=1).index.to_numpy())
        imgs = f["x"][idx]
        for c in range(8):
            axes[r, c].imshow(imgs[c]); axes[r, c].axis("off")
        axes[r, 0].set_title(f"{h} {'tumour' if lab else 'normal'}", loc="left", fontsize=10)
plt.tight_layout(); plt.savefig(f"{OUT}/grid_by_hospital.png", dpi=80); plt.show()

In [4]:
%cd /content
!rm -rf calib-cl
!git clone -q https://github.com/safia-ahmad/calib-cl.git
%cd /content/calib-cl
from google.colab import drive
drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/calib-cl-data"
!mkdir -p /content/data
!cp {DRIVE}/pcam_compact.h5 {DRIVE}/pcam_compact_manifest.csv /content/data/
!ls -lh /content/data

/content
/content/calib-cl
Mounted at /content/drive
total 2.9G
-rw------- 1 root root 2.9G Oct 29 17:19 pcam_compact.h5
-rw------- 1 root root  11M Oct 29 17:19 pcam_compact_manifest.csv


In [ ]:
!pip install -q pytest
!python -m pytest -q


In [ ]:
import time, numpy as np, pandas as pd
from calcl.data import load_compact, slide_table, indices_for, PatchDataset, mean_colour
t0 = time.time()
X, man = load_compact("/content/data/pcam_compact.h5", "/content/data/pcam_compact_manifest.csv")
print(f"loaded {X.shape} in {time.time()-t0:.0f}s, {X.nbytes/1e9:.1f} GB in RAM")
st = slide_table(man)
display(pd.crosstab(st.hospital, st.slide_type, margins=True))
!free -h | head -2

In [ ]:
import torch
from torch.utils.data import DataLoader
umcu_slides = st[st.hospital == "UMCU"].wsi
idx = indices_for(man, umcu_slides)
ds = PatchDataset(X, man.label.to_numpy(), idx, train=True)
dl = DataLoader(ds, batch_size=64, shuffle=True, num_workers=2)
t0, n = time.time(), 0
for b, (x, y) in enumerate(dl):
    n += len(y)
    if b == 49: break
print("batch:", x.shape, x.dtype, "| labels in batch:", y.tolist()[:10])
print(f"{n/(time.time()-t0):.0f} patches per second")

In [ ]:
rows = []
for h in ["RUMC", "UMCU"]:
    for lab in [0, 1]:
        ii = indices_for(man, st[st.hospital == h].wsi, label=lab)
        r, g, b = mean_colour(X, ii)
        rows.append({"hospital": h, "class": "tumour" if lab else "normal",
                     "mean_R": round(r, 1), "mean_G": round(g, 1), "mean_B": round(b, 1)})
colour = pd.DataFrame(rows)
display(colour)
colour.to_csv(f"{DRIVE}/colour_by_hospital.csv", index=False)

In [ ]:
%cd /content
!rm -rf calib-cl
!git clone -q https://github.com/safia-ahmad/calib-cl.git
%cd /content/calib-cl
from google.colab import drive
drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/calib-cl-data"
!mkdir -p /content/data
!cp -n {DRIVE}/pcam_compact.h5 {DRIVE}/pcam_compact_manifest.csv /content/data/
!pip install -q pytest timm
!python -m pytest -q
!nvidia-smi --query-gpu=name,memory.total --format=csv

/content
/content/calib-cl
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
.............................                                            [100%]
29 passed in 13.75s
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
import pandas as pd
from calcl.data import load_compact, slide_table
from calcl.splits import make_split, summarise
X, man = load_compact("/content/data/pcam_compact.h5", "/content/data/pcam_compact_manifest.csv")
st = slide_table(man)
import os; os.makedirs(f"{DRIVE}/splits", exist_ok=True)
for setting in ["pcam_fwd", "pcam_rev"]:
    for seed in range(6):
        make_split(st, setting, seed).to_csv(f"{DRIVE}/splits/{setting}_seed{seed}.csv", index=False)
print("12 split files saved")
display(summarise(make_split(st, "pcam_fwd", 1)))
a, b = make_split(st, "pcam_fwd", 1), make_split(st, "pcam_fwd", 2)
print("slides whose role changes between seed 1 and seed 2:", (a.set_index("wsi").role != b.set_index("wsi").role).sum())

12 split files saved


slide_type       normal  tumour  All
domain role                         
source stream        20      14   34
       test          20      14   34
       train         45      32   77
       val_buf        5       3    8
       val_es         5       4    9
       val_temp       5       3    8
target dev            6       4   10
       stream        35      24   59
       test          18      12   30
All                 159     110  269

slides whose role changes between seed 1 and seed 2: 165


In [ ]:
import torch, time
from calcl.models import make_model
from calcl.data import PatchDataset
from torch.utils.data import DataLoader
dev = torch.device("cuda")
dl = DataLoader(PatchDataset(X, man.label.to_numpy(), list(range(128))), batch_size=128)
x, y = next(iter(dl))
for name in ["resnet18", "vit_s16"]:
    m = make_model(name, pretrained=True).to(dev).eval()
    with torch.no_grad():
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(5): out = m(x.to(dev))
        torch.cuda.synchronize()
    print(f"{name}: output {tuple(out.shape)}, {sum(p.numel() for p in m.parameters())/1e6:.0f}M params, "
          f"{5*128/(time.time()-t0):.0f} patches/s (inference)")

resnet18: output (128, 2), 11M params, 2947 patches/s (inference)
vit_s16: output (128, 2), 22M params, 317 patches/s (inference)


In [ ]:
!python scripts/train_source.py --setting pcam_fwd --seed 0 --model resnet18 \
    --data /content/data --out /content/data/test_runs --epochs 1 --max_train 3000

device: cuda | run folder: /content/data/test_runs/pcam_fwd_resnet18_seed0
slide_type       normal  tumour  All
domain role                         
source stream        20      14   34
       test          20      14   34
       train         45      32   77
       val_buf        5       3    8
       val_es         5       4    9
       val_temp       5       3    8
target dev            6       4   10
       stream        35      24   59
       test          18      12   30
All                 159     110  269
patches: train 3000, val_es 4285, val_temp 3781, S 1840, G 1720
epoch 1: train loss 0.3754 | val NLL 0.5747 | val acc 0.734 | 5s
temperature T0 = 1.708

=== results (temperature-scaled) ===
S_source_test  acc 0.796  AUROC 0.899  Brier 0.1366  MCB -0.0035  DSC 0.1080  UNC 0.2481
G_target_test  acc 0.732  AUROC 0.873  Brier 0.1737  MCB 0.0325  DSC 0.1022  UNC 0.2434
G4: Brier(G) - Brier(S) = 0.0371 -> PASS (needs >= 0.005)
total time 0.6 min


In [ ]:
!python scripts/train_source.py --setting pcam_fwd --seed 0 --model resnet18 \
    --data /content/data --out {DRIVE}/runs

device: cuda | run folder: /content/drive/MyDrive/calib-cl-data/runs/pcam_fwd_resnet18_seed0
slide_type       normal  tumour  All
domain role                         
source stream        20      14   34
       test          20      14   34
       train         45      32   77
       val_buf        5       3    8
       val_es         5       4    9
       val_temp       5       3    8
target dev            6       4   10
       stream        35      24   59
       test          18      12   30
All                 159     110  269
patches: train 37070, val_es 4285, val_temp 3781, S 1840, G 1720
epoch 1: train loss 0.2033 | val NLL 0.2507 | val acc 0.899 | 19s
epoch 2: train loss 0.1196 | val NLL 0.2399 | val acc 0.910 | 18s
epoch 3: train loss 0.0940 | val NLL 0.1972 | val acc 0.921 | 17s
epoch 4: train loss 0.0787 | val NLL 0.2357 | val acc 0.913 | 17s
epoch 5: train loss 0.0676 | val NLL 0.2425 | val acc 0.914 | 17s
epoch 6: train loss 0.0587 | val NLL 0.2492 | val acc 0.912 | 17s
ea

In [ ]:
!python scripts/diagnose_run.py --run /content/drive/MyDrive/calib-cl-data/runs/pcam_fwd_resnet18_seed0

run: /content/drive/MyDrive/calib-cl-data/runs/pcam_fwd_resnet18_seed0
T0 = 2.202, best epoch 3

== S_source_test (1840 patches, 34 slides)
  AUROC 0.943 | share predicted tumour 0.38 (truth 0.46)
  mean p(tumour): true normal 0.130 | true tumour 0.743
  accuracy: on normal patches 0.964 | on tumour patches 0.799
  5 worst slides:
                              n  tumour       acc
wsi                                              
camelyon16_train_tumor_012   98      60  0.500000
camelyon16_train_tumor_050   93      60  0.591398
camelyon16_train_tumor_008   88      60  0.738636
camelyon16_train_tumor_024   91      60  0.758242
camelyon16_train_normal_057  31       0  0.838710

== G_target_test (1720 patches, 30 slides)
  AUROC 0.630 | share predicted tumour 0.76 (truth 0.42)
  mean p(tumour): true normal 0.595 | true tumour 0.711
  accuracy: on normal patches 0.317 | on tumour patches 0.864
  5 worst slides:
                              n  tumour       acc
wsi                           

In [ ]:
!python scripts/train_source.py --setting pcam_fwd --seed 0 --model resnet18 --colour 0.5
!python scripts/diagnose_run.py --run /content/drive/MyDrive/calib-cl-data/runs/pcam_fwd_resnet18_seed0_colour0.5

device: cuda | run folder: /content/drive/MyDrive/calib-cl-data/runs/pcam_fwd_resnet18_seed0_colour0.5
slide_type       normal  tumour  All
domain role                         
source stream        20      14   34
       test          20      14   34
       train         45      32   77
       val_buf        5       3    8
       val_es         5       4    9
       val_temp       5       3    8
target dev            6       4   10
       stream        35      24   59
       test          18      12   30
All                 159     110  269
patches: train 37070, val_es 4285, val_temp 3781, S 1840, G 1720
epoch 1: train loss 0.2256 | val NLL 0.3071 | val acc 0.861 | 37s
epoch 2: train loss 0.1479 | val NLL 0.2399 | val acc 0.893 | 37s
epoch 3: train loss 0.1268 | val NLL 0.1941 | val acc 0.914 | 34s
epoch 4: train loss 0.1041 | val NLL 0.2337 | val acc 0.898 | 35s
epoch 5: train loss 0.0947 | val NLL 0.1995 | val acc 0.911 | 38s
epoch 6: train loss 0.0847 | val NLL 0.2417 | val acc 0.90

In [5]:
!python scripts/validate_v1.py --run /content/drive/MyDrive/calib-cl-data/runs/pcam_fwd_resnet18_seed0_colour0.5 --data /content/drive/MyDrive/calib-cl-data

run: /content/drive/MyDrive/calib-cl-data/runs/pcam_fwd_resnet18_seed0_colour0.5
T0 = 1.764, 1840 patches, 34 slides

check                       damage  brier     mcb    dsc   d_mcb   d_dsc pass
    -              none (baseline) 0.0736 -0.0121 0.1624  0.0000  0.0000     
    A                  logit / 0.5 0.0726 -0.0130 0.1624 -0.0010  0.0000 FAIL
    A                    logit / 2 0.0973  0.0116 0.1624  0.0237 -0.0000 PASS
    A                    logit / 3 0.1229  0.0372 0.1624  0.0493 -0.0000 PASS
    A                     logit -1 0.1012  0.0156 0.1624  0.0276 -0.0000 PASS
    A                     logit +1 0.0829 -0.0028 0.1624  0.0093  0.0000 PASS
    B               10% set to 0.5 0.0913 -0.0101 0.1467  0.0020 -0.0158 PASS
    B               30% set to 0.5 0.1267 -0.0085 0.1129  0.0036 -0.0495 PASS
    B               50% set to 0.5 0.1614 -0.0062 0.0805  0.0058 -0.0819 PASS
    C logit / 2 and 30% set to 0.5 0.1428  0.0074 0.1128  0.0195 -0.0497 PASS

V1: FAIL (8 of 9 damage

In [9]:
!python scripts/validate_v1.py --run /content/drive/MyDrive/calib-cl-data/runs/pcam_fwd_resnet18_seed0_colour0.5 --data /content/drive/MyDrive/calib-cl-data

run: /content/drive/MyDrive/calib-cl-data/runs/pcam_fwd_resnet18_seed0_colour0.5
T0 = 1.764, 1840 patches, 34 slides
calibrating start on S: logit -> 1.434 * logit + +0.588 (underconfident at T0)

check                       damage  brier     mcb    dsc  d_mcb   d_dsc pass
    -              none (baseline) 0.0685 -0.0171 0.1624 0.0000  0.0000     
    A                  logit / 0.5 0.0713 -0.0144 0.1624 0.0027  0.0000 PASS
    A                    logit / 2 0.0815 -0.0042 0.1624 0.0129 -0.0000 PASS
    A                    logit / 3 0.1011  0.0155 0.1624 0.0326 -0.0000 PASS
    A                     logit -1 0.0775 -0.0081 0.1624 0.0090 -0.0000 PASS
    A                     logit +1 0.0836 -0.0021 0.1624 0.0150  0.0000 PASS
    B               10% set to 0.5 0.0868 -0.0132 0.1481 0.0039 -0.0144 PASS
    B               30% set to 0.5 0.1230 -0.0108 0.1143 0.0064 -0.0481 PASS
    B               50% set to 0.5 0.1586 -0.0081 0.0814 0.0090 -0.0810 PASS
    C logit / 2 and 30% set to 0.